In [ ]:
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
import torch
import gensim.downloader
import pandas as pd
import spacy
from collections import defaultdict
import random
import json
import nltk
from nltk.corpus import wordnet as wn
from collections import Counter
from itertools import combinations

In [22]:
def load_data(path):
    dataset = load_dataset("wikitext", "wikitext-103-v1", split="train")

    glove = gensim.downloader.load("glove-wiki-gigaword-50")
    glove_voc = set(glove.key_to_index.keys())

    df = pd.read_csv(path, 
                sep="\t", 
                names=["lemma", "forms", "fam", "transf"])
    return dataset, glove_voc, df

In [23]:
def clean_dataset(dataset):
    content = []
    for line in dataset["text"]:
        line = line.strip().lower()
        if line and not line.startswith("=") and len(line.split()) > 4:
            content.append(line)
    return content

In [24]:
def get_target_words(df, glove_voc):
    target = {
            word.strip().lower()
            for form_list in df["forms"].str.split(",")
            for word in form_list
            if word.strip().lower() in glove_voc
    }
    return target

In [25]:
def get_matched_sent(content, target, json_path=None):
    if json_path:
        return pd.read_json(json_path)

    nlp = spacy.blank("en")
    nlp.add_pipe("sentencizer")
    
    matched = defaultdict(set)

    for doc in nlp.pipe(content, batch_size=2048, n_process=2):
        for sent in doc.sents:
            if len(sent) > 40:
                continue

            sent_text = sent.text
            tokens = {token.text for token in sent}

            for word in target.intersection(tokens):
                matched[word].add(sent_text)

    def tag_word(sents):
        n = len(sents)
        if n >= 50:
            return "sampled"
        elif n >= 10:
            return "low_freq"
        return "excluded"

    def sample_sents(sents):
        sents = list(sents)
        n = len(sents)
        if n >= 50:
            return random.sample(sents, 50)
        return sents

    random.seed(42)
    sent_df = pd.DataFrame(matched.items(),
                            columns=["word", "sents"])

    sent_df["tag"] = sent_df["sents"].apply(tag_word)
    sent_df["sents"] = sent_df["sents"].apply(sample_sents)

    sent_df.to_json("sent_df.json")
    print("sent_df json created")
    return sent_df

In [26]:
dataset, glove_voc, df = load_data("/home/onyxia/work/morph_families.tsv")

print("content..")
content = clean_dataset(dataset)
del dataset

print("target..")
target = get_target_words(df, glove_voc)
del df

print("matched..")
sent_df = get_matched_sent(content, target, json_path="/home/onyxia/work/sent_df.json")

content..
target..
matched..


In [27]:
sent_df.sample()

,word,sents,tag
5,opening,[the netherlands qualified for the 2006 fifa w...,sampled


In [ ]:
def get_synonyms():
    print("get synonyms..")
    synonyms = set()
    for synset in wn.all_synsets():
        lemmas = [l.name().lower().replace("_", " ") for l in synset.lemmas()]
        for l1, l2 in combinations(lemmas, 2):
            res = tuple(sorted((l1, l2)))
            synonyms.add(res)

    with open("/home/onyxia/work/synonyms.json", "w") as f:
        json.dump(list(synonyms), f)

    return list(synonyms)


def get_antonyms():
    print("get antonyms..")
    antonyms = set()
    for lemma in wn.all_lemmas():
        word = lemma.name().lower().replace("_", " ")
        for ant in lemma.antonyms():
            word_ant = ant.name().lower().replace("_", " ")
            res = tuple(sorted((word, word_ant)))
            antonyms.add(res)

    with open("/home/onyxia/work/antonyms.json", "w") as f:
        json.dump(list(antonyms), f)

    return list(antonyms)


def filter_word(pair_set, content, glove_voc, filename):
    print("filter words..")
    random.seed(42)
    filtered = []
    word_counts = Counter()

    for sentence in content:
        word_counts.update(sentence.split())

    for l1, l2 in pair_set:
        if l1 in glove_voc and l2 in glove_voc:
            if word_counts[l1] >= 10 and word_counts[l2] >= 10:
                filtered.append((l1, l2))

    if len(filtered) > 200:
        filtered = random.sample(filtered, 200)
    else:
        print(f"{len(filtered)} pairs found")
    
    with open(f"/home/onyxia/work/{filename}.json", "w") as f:
        json.dump(filtered, f)
        
    return list(filtered)

nltk.download("wordnet")
final_synonyms = filter_word(get_synonyms(), content, glove_voc, filename="synonyms")
final_antonyms = filter_word(get_antonyms(), content, glove_voc, filename="antonyms")

[nltk_data] Downloading package wordnet to /home/onyxia/nltk_data...


KeyboardInterrupt: 